In [1]:
from src.data import load_all

corpus, queries, qrels = load_all(split= "test")


/Users/nourlachtar/projets/projet/scifact-verifier/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-14 21:44:42 | INFO     | src.data.load_scifact | Loading corpus from cache.
2026-05-14 21:44:42 | INFO     | src.data.load_scifact | Loading qrels for split 'test' from cache.
2026-05-14 21:44:42 | INFO     | src.data.load_scifact | Loading queries from cache.
2026-05-14 21:44:42 | INFO     | src.data.load_scifact | Loaded split 'test': 300 queries, 339 qrels entries, 5183 corpus documents.


In [2]:
from src.retrieval import BM25Retriever, DenseRetriever, HybridRetriever , RerankedRetriever

In [3]:
bm25_retriever = BM25Retriever()
bm25_retriever.load()

2026-05-14 21:44:44 | INFO     | src.retrieval.bm25_retriever | Loading BM25 index (scenario='default') …
2026-05-14 21:44:44 | INFO     | src.retrieval.bm25_retriever | BM25 index loaded: 5183 documents, scenario='default'.


True

In [4]:
dense_retriever = DenseRetriever()
dense_retriever.load()

2026-05-14 21:44:44 | INFO     | src.retrieval.dense_retriever | Loading FAISS index …
2026-05-14 21:44:44 | INFO     | src.retrieval.dense_retriever | Dense index loaded (5183 vectors, dim=768, model=allenai/specter).


True

In [7]:
hybrid_retriever = HybridRetriever([bm25_retriever, dense_retriever])
hybrid_retriever.load()

2026-05-14 21:44:58 | INFO     | src.retrieval.bm25_retriever | Loading BM25 index (scenario='default') …
2026-05-14 21:44:58 | INFO     | src.retrieval.bm25_retriever | BM25 index loaded: 5183 documents, scenario='default'.
2026-05-14 21:44:58 | INFO     | src.retrieval.dense_retriever | Loading FAISS index …
2026-05-14 21:44:58 | INFO     | src.retrieval.dense_retriever | Dense index loaded (5183 vectors, dim=768, model=allenai/specter).


True

In [8]:
reranked_retriever = RerankedRetriever(hybrid_retriever,corpus=corpus)
reranked_retriever.load()

2026-05-14 21:44:58 | INFO     | src.retrieval.bm25_retriever | Loading BM25 index (scenario='default') …
2026-05-14 21:44:58 | INFO     | src.retrieval.bm25_retriever | BM25 index loaded: 5183 documents, scenario='default'.
2026-05-14 21:44:58 | INFO     | src.retrieval.dense_retriever | Loading FAISS index …
2026-05-14 21:44:58 | INFO     | src.retrieval.dense_retriever | Dense index loaded (5183 vectors, dim=768, model=allenai/specter).


True

In [9]:
from src.evaluation.evaluate_retrieval import evaluate_retrieval

retrievers = {
    "BM25": bm25_retriever,
    "Dense": dense_retriever,
    "Hybrid": hybrid_retriever,
    "Reranked": reranked_retriever,
}

report = {}

for name, retriever in retrievers.items():
    result = evaluate_retrieval(
        name=name,
        retriever=retriever,
        queries=queries,
        qrels=qrels,
        top_k=10,
    )

    report[name] = result
    print(f"\nResults for {name}:")
    for metric, value in result.items():
        print(f"  {metric}: {value}")

BM25: 100%|██████████| 300/300 [00:01<00:00, 265.32query/s]



Results for BM25:
  Recall@1: 0.5244
  Recall@5: 0.7386
  Recall@10: 0.8112
  P@5: 0.158
  MRR: 0.6341
  nDCG@10: 0.6737
  latency_ms_per_query: 3.8
  top_k: 10


Dense:   0%|          | 0/300 [00:00<?, ?query/s]

2026-05-14 21:45:04 | INFO     | src.retrieval.dense_retriever | Loading embedding model: allenai/specter (device=mps) …


Dense: 100%|██████████| 300/300 [00:05<00:00, 55.88query/s] 


...


Results for Hybrid:
  Recall@1: 0.4501
  Recall@5: 0.7415
  Recall@10: 0.8101
  P@5: 0.1607
  MRR: 0.589
  nDCG@10: 0.6374
  latency_ms_per_query: 13.9
  top_k: 10


Reranked:   0%|          | 0/300 [00:00<?, ?query/s]

2026-05-14 21:45:13 | INFO     | src.retrieval.reranked_retriever | Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 …


Reranked: 100%|██████████| 300/300 [02:31<00:00,  1.98query/s]


Results for Reranked:
  Recall@1: 0.5543
  Recall@5: 0.7549
  Recall@10: 0.8312
  P@5: 0.1667
  MRR: 0.6683
  nDCG@10: 0.7
  latency_ms_per_query: 504.8
  top_k: 10
